# 07: Deterministic Preprocessing, Signal Normalization & Feature Store

## 1. Research Scope & Objectives
This notebook benchmarks the complete deterministic preprocessing pipeline (pre-emphasis, VAD, 4.0-second fixed windowing, peak normalization) and verifies PyTorch lazy loading throughput.

### Pipeline Transformations
1. Mono channel standardization at 16,000 Hz
2. Pre-emphasis: $y[n] = x[n] - 0.97 x[n-1]$
3. VAD energy trimming
4. Fixed-length cropping/padding to $L = 64,000\text{ samples}$
5. Peak amplitude scaling to $[-1.0, 1.0]$

In [1]:
import os, sys
sys.path.insert(0, os.path.abspath(".."))

import torch, numpy as np, pandas as pd, matplotlib.pyplot as plt, time
from src.features.preprocessor import SignalPreprocessor
from src.features.spectral import SpectralFeatureExtractor
from src.features.cepstral import CepstralFeatureExtractor
from src.data.dataset import create_dataloader
from src.utils.config import path_config

manifest_path = os.path.join(path_config.data_processed, "protocol_manifest.parquet")
df = pd.read_parquet(manifest_path)

In [2]:
preprocessor = SignalPreprocessor()
spectral_ext = SpectralFeatureExtractor()
cepstral_ext = CepstralFeatureExtractor()

sample_df = df[df.split == "train"].sample(200, random_state=42)
train_loader_lfcc = create_dataloader(sample_df, mode="train", feature_type="lfcc", batch_size=16)
train_loader_mel = create_dataloader(sample_df, mode="train", feature_type="mel", batch_size=16)
train_loader_raw = create_dataloader(sample_df, mode="train", feature_type="raw", batch_size=16)

In [3]:
for batch_feat, batch_label in train_loader_lfcc:
    print("LFCC Batch Feature Shape:", batch_feat.shape, "Labels Shape:", batch_label.shape)
    break
for batch_feat, batch_label in train_loader_mel:
    print("Log-Mel Batch Feature Shape:", batch_feat.shape, "Labels Shape:", batch_label.shape)
    break
for batch_feat, batch_label in train_loader_raw:
    print("Raw Waveform Batch Feature Shape:", batch_feat.shape, "Labels Shape:", batch_label.shape)
    break

c:\Users\tamimystic\.conda\envs\ml\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


LFCC Batch Feature Shape: torch.Size([16, 1, 60, 247]) Labels Shape: torch.Size([16])
Log-Mel Batch Feature Shape: torch.Size([16, 1, 128, 251]) Labels Shape: torch.Size([16])
Raw Waveform Batch Feature Shape: torch.Size([16, 1, 64000]) Labels Shape: torch.Size([16])


In [4]:
t0 = time.time()
total_samples = 0
for batch_feat, _ in train_loader_lfcc:
    total_samples += batch_feat.size(0)
elapsed = time.time() - t0
print(f"Processed {total_samples} audio utterances in {elapsed:.2f}s ({total_samples/elapsed:.1f} utterances/sec on CPU)")

Processed 200 audio utterances in 1.52s (131.6 utterances/sec on CPU)
